# LeetCode 1554 — Strings Differ by One Character bằng Python

Link đề: https://leetcode.com/problems/strings-differ-by-one-character/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Rolling Hash, Hash Function

> **Ghi chú Premium:** đây là bài Premium (🔒). Đề bài và chữ ký hàm dưới đây được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1554/). Nếu có tài khoản Premium, nên đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
class Solution:
    def differByOne(self, dict: List[str]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu khái niệm **khoảng cách Hamming** giữa hai chuỗi cùng độ dài và cài đặt baseline so mọi cặp `O(n²·m)`.
2. Nắm kỹ thuật **"ký tự đại diện" (wildcard)**: thay từng vị trí bằng `'*'` để biến câu hỏi "khác đúng 1 ký tự" thành câu hỏi "trùng khoá trong `set`".
3. Hiểu **rolling hash / polynomial hash**: tính hash của chuỗi "bỏ đi ký tự thứ `j`" trong `O(1)` để đạt `O(n·m)`, và biết cách xử lý **va chạm hash**.
4. Biết phân tích độ phức tạp khi ràng buộc cho **tổng số ký tự** (`n·m ≤ 10⁵`) thay vì `n` và `m` riêng lẻ.
5. Tránh các bẫy: so một từ với chính nó, bỏ ký tự mà không ghi nhớ vị trí, kiểm tra `set` sau khi đã thêm.

## 2. Đề bài & ví dụ

Cho danh sách chuỗi `dict`, mọi chuỗi có **cùng độ dài**. Trả về `True` nếu tồn tại **hai chuỗi chỉ khác nhau đúng 1 ký tự ở cùng một chỉ số**, ngược lại trả `False`.

**Ví dụ 1:** `dict = ["abcd","acbd","aacd"]` → `True`. `"abcd"` và `"aacd"` chỉ khác nhau ở chỉ số 1.

**Ví dụ 2:** `dict = ["ab","cd","yz"]` → `False`.

**Ví dụ 3:** `dict = ["abcd","cccc","abyd","abab"]` → `True` (`"abcd"` và `"abyd"` khác nhau ở chỉ số 2).

**Ràng buộc:**
- Tổng số ký tự trong `dict` ≤ `10⁵`.
- `dict[i].length == dict[j].length`.
- Các `dict[i]` đôi một khác nhau.
- `dict[i]` chỉ gồm chữ cái tiếng Anh thường.

*(Follow-up thường được nhắc: giải trong `O(n·m)` với `n` = số chuỗi, `m` = độ dài mỗi chuỗi.)*

In [1]:
import random
import string
import time
from collections import Counter, defaultdict
from typing import List

examples = [
    (["abcd", "acbd", "aacd"], True),
    (["ab", "cd", "yz"], False),
    (["abcd", "cccc", "abyd", "abab"], True),
]
for words, want in examples:
    print(f"dict = {words} -> mong đợi {want}")

dict = ['abcd', 'acbd', 'aacd'] -> mong đợi True
dict = ['ab', 'cd', 'yz'] -> mong đợi False
dict = ['abcd', 'cccc', 'abyd', 'abab'] -> mong đợi True


## 3. Kiến thức nền

### 3.1 Khoảng cách Hamming với `zip`

Hai chuỗi cùng độ dài: đếm số vị trí ký tự khác nhau. `zip(a, b)` ghép từng cặp ký tự.

In [2]:
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

print(hamming("abcd", "aacd"), hamming("abcd", "acbd"), hamming("ab", "cd"))

1 2 2


### 3.2 Cắt chuỗi (slicing) và ghép chuỗi

`w[:i]` là phần trước vị trí `i`, `w[i+1:]` là phần sau. `w[:i] + "*" + w[i+1:]` thay ký tự thứ `i` bằng `'*'`. Chuỗi Python bất biến nên phép này **tạo chuỗi mới** độ dài `m` — tốn `O(m)`.

In [3]:
w = "abcd"
for i in range(len(w)):
    print(i, w[:i] + "*" + w[i + 1:])

0 *bcd
1 a*cd
2 ab*d
3 abc*


### 3.3 `set` — tra cứu trung bình `O(1)`

`x in s` và `s.add(x)` trên `set` mất trung bình `O(1)` phép băm (nhưng băm một chuỗi dài `m` tốn `O(m)`).

In [4]:
s = set()
for t in ["a*cd", "ab*d", "a*cd"]:
    print(t, "đã có" if t in s else "chưa có")
    s.add(t)

a*cd chưa có
ab*d chưa có
a*cd đã có


### 3.4 `ord` và hash đa thức

`ord(c) - ord('a')` đổi chữ thành số `0..25`. Hash đa thức của chuỗi `c₀c₁...c_{m−1}`:

`H = c₀·B^{m−1} + c₁·B^{m−2} + ... + c_{m−1}·B⁰ (mod P)`.

Bỏ đi đóng góp của vị trí `j` (thay bằng "rỗng/0"): `H − c_j·B^{m−1−j}`. Hai chuỗi chỉ khác nhau ở vị trí `j` sẽ có cùng giá trị này.

In [5]:
B, P = 27, (1 << 61) - 1

def poly_hash(w):
    h = 0
    for ch in w:
        h = (h * B + (ord(ch) - 96)) % P      # 'a' -> 1 ... 'z' -> 26 (tránh chữ 'a' = 0)
    return h

def hash_without(w, j):
    m = len(w)
    return (poly_hash(w) - (ord(w[j]) - 96) * pow(B, m - 1 - j, P)) % P

print(hash_without("abcd", 2), hash_without("abyd", 2), hash_without("abcd", 1))

21145 21145 19768


### 3.5 `Counter` và `defaultdict(list)`

`Counter` đếm số lần mỗi khoá xuất hiện; `defaultdict(list)` gom nhóm phần tử theo khoá. Dùng trong biến thể "đếm số cặp" và để xác minh va chạm hash.

In [6]:
cnt = Counter(["a*c", "a*c", "*bc"])
print(cnt)
groups = defaultdict(list)
for idx, key in enumerate([5, 7, 5]):
    groups[key].append(idx)
print(dict(groups))

Counter({'a*c': 2, '*bc': 1})
{5: [0, 2], 7: [1]}


## 4. Ý tưởng nền tảng / trực giác

Lấy ví dụ 1: `["abcd", "acbd", "aacd"]`. Tính tay khoảng cách Hamming mọi cặp:

| Cặp | So từng ký tự | Hamming |
|-----|---------------|---------|
| abcd – acbd | a=a, b≠c, c≠b, d=d | 2 |
| abcd – aacd | a=a, **b≠a**, c=c, d=d | **1** ✔ |
| acbd – aacd | a=a, c≠a, b≠c, d=d | 2 |

Có cặp Hamming = 1 ⇒ `True`.

**Trực giác cho cách nhanh:** nếu `"abcd"` và `"aacd"` chỉ khác nhau ở vị trí 1, thì khi **che** vị trí 1 lại, cả hai thành `"a*cd"` — giống hệt nhau. Ngược lại nếu hai từ khi cùng che vị trí `i` trở nên giống nhau thì chúng chỉ có thể khác nhau ở vị trí `i` (và vì từ khác nhau nên khác đúng ở `i`).

In [7]:
words = ["abcd", "acbd", "aacd"]
for w in words:
    print(w, "->", [w[:i] + "*" + w[i + 1:] for i in range(len(w))])

abcd -> ['*bcd', 'a*cd', 'ab*d', 'abc*']
acbd -> ['*cbd', 'a*bd', 'ac*d', 'acb*']
aacd -> ['*acd', 'a*cd', 'aa*d', 'aac*']


## 5. Cách 1 — Brute Force / baseline: so mọi cặp

**Ý tưởng.** Với mỗi cặp `(i, j)`, `i < j`, đếm số vị trí khác nhau; nếu bằng 1 thì trả `True`. Có thể dừng sớm khi đếm vượt 1.

**Độ phức tạp:** `O(n²·m)` time, `O(1)` space. Với `n·m ≤ 10⁵`, trường hợp xấu nhất `m = 1`, `n = 10⁵` cho `~5·10⁹` phép so — quá chậm (tuy nhiên `m = 1` thì mọi cặp đều khác 1 ký tự nên dừng ngay; xấu thật sự là khi đáp án `False` với `n` lớn, ví dụ `m = 4`, `n = 25 000`).

In [8]:
def brute_pairs(words):
    n = len(words)
    for i in range(n):
        for j in range(i + 1, n):
            diff = 0
            for a, b in zip(words[i], words[j]):
                if a != b:
                    diff += 1
                    if diff > 1:
                        break
            if diff == 1:
                return True
    return False

for words, want in examples:
    print(brute_pairs(words), "mong đợi", want)

True mong đợi True
False mong đợi False
True mong đợi True


Code nộp LeetCode cho cách 1 (sẽ TLE với test lớn, chỉ để tham khảo):

In [9]:
class SolutionBrute:
    def differByOne(self, dict: List[str]) -> bool:
        words = dict
        for i in range(len(words)):
            for j in range(i + 1, len(words)):
                if sum(a != b for a, b in zip(words[i], words[j])) == 1:
                    return True
        return False

print([SolutionBrute().differByOne(w) for w, _ in examples])

[True, False, True]


In [10]:
# Sinh bộ từ chắc chắn KHÔNG có cặp nào khác đúng 1 ký tự:
# ký tự cuối là "checksum" = tổng các ký tự trước (mod 26). Hai từ khác nhau ở đúng 1 vị trí đầu
# thì checksum cũng khác -> khác ít nhất 2 vị trí.
def no_pair_words(n, m, rng):
    words = set()
    while len(words) < n:
        body = [rng.randrange(26) for _ in range(m - 1)]
        body.append(sum(body) % 26)
        words.add("".join(chr(97 + v) for v in body))
    return list(words)

rng = random.Random(0)
for n in [300, 600, 1200]:
    words = no_pair_words(n, 8, rng)
    t0 = time.perf_counter()
    r = brute_pairs(words)
    print(f"n={n}: đáp án {r}, so mọi cặp mất {time.perf_counter() - t0:.3f}s")

n=300: đáp án False, so mọi cặp mất 0.011s
n=600: đáp án False, so mọi cặp mất 0.046s


n=1200: đáp án False, so mọi cặp mất 0.208s


**Nút thắt:** ta so **mọi cặp** dù đa số cặp khác nhau rất nhiều. Cần cách "gom nhóm" những từ có khả năng khác đúng 1 ký tự mà không so từng cặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (che vị trí).** Với mỗi từ `w` và mỗi vị trí `i`, định nghĩa *mẫu* `pattern(w, i) = w[:i] + "*" + w[i+1:]`.

- *Nếu* `u` và `v` khác nhau đúng ở vị trí `i` *thì* `pattern(u, i) == pattern(v, i)`.
- *Nếu* `pattern(u, i) == pattern(v, j)` với `u ≠ v` *thì* dấu `*` cùng chỗ nên `i = j`, các ký tự còn lại giống nhau ⇒ `u`, `v` khác nhau **đúng** ở vị trí `i`.

Vậy: bài toán ⇔ "có hai từ sinh ra cùng một mẫu không?" ⇒ duyệt các mẫu, lưu vào `set`, gặp mẫu đã có ⇒ `True`.

**Insight 2 (băm nhanh).** Tạo mẫu bằng slicing tốn `O(m)` mỗi mẫu ⇒ `O(n·m²)`. Thay vì tạo chuỗi, ta dùng **hash đa thức**: hash của "từ bỏ ký tự `j`" = `H(w) − c_j·B^{m−1−j}` tính trong `O(1)` khi đã có `H(w)` và bảng luỹ thừa ⇒ tổng `O(n·m)`. Khoá trong `set` là cặp `(j, giá trị hash)` (thật ra giá trị đã chứa thông tin vị trí, nhưng thêm `j` cho chắc).

**Invariant khi duyệt:** `set` chứa mẫu của mọi (từ, vị trí) đã xét trước đó; mẫu hiện tại có trong `set` ⇔ có từ trước khác từ hiện tại đúng một ký tự (ở vị trí của mẫu).

In [11]:
u, v = "abcd", "abyd"
print([(i, u[:i] + "*" + u[i+1:], v[:i] + "*" + v[i+1:]) for i in range(4)])
print("chỉ trùng ở i = 2")

[(0, '*bcd', '*byd'), (1, 'a*cd', 'a*yd'), (2, 'ab*d', 'ab*d'), (3, 'abc*', 'aby*')]
chỉ trùng ở i = 2


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Wildcard `'*'` + `set` (lời giải doocs)

Từng đoạn:
1. `s = set()` — tập các mẫu đã thấy.
2. Với mỗi `word`, mỗi `i`: `t = word[:i] + "*" + word[i+1:]`.
3. Nếu `t in s` ⇒ có từ trước sinh cùng mẫu ⇒ `True`. Ngược lại thêm `t` vào `s`.
4. Hết vòng ⇒ `False`.

Lưu ý: một từ không thể tự trùng mẫu với chính nó (các mẫu của cùng từ có `*` ở chỗ khác nhau), nên kiểm tra rồi thêm ngay trong cùng vòng là an toàn.

In [12]:
def wildcard_set(words):
    s = set()
    for word in words:
        for i in range(len(word)):
            t = word[:i] + "*" + word[i + 1:]
            if t in s:
                return True
            s.add(t)
    return False

for words, want in examples:
    print(wildcard_set(words), "mong đợi", want)

True mong đợi True
False mong đợi False
True mong đợi True


Code nộp LeetCode cho cách 2 (đúng chữ ký doocs, tham số tên `dict`):

In [13]:
class SolutionWildcard:
    def differByOne(self, dict: List[str]) -> bool:
        s = set()
        for word in dict:
            for i in range(len(word)):
                t = word[:i] + "*" + word[i + 1 :]
                if t in s:
                    return True
                s.add(t)
        return False

print([SolutionWildcard().differByOne(w) for w, _ in examples])

[True, False, True]


**Độ phức tạp:** `n·m` mẫu, mỗi mẫu dài `m` (tạo + băm `O(m)`) ⇒ Time `O(n·m²)`, Space `O(n·m²)` ký tự trong `set`. Khi `m` nhỏ (thường gặp) thì rất nhanh; khi `m` lớn (ví dụ 2 chuỗi dài `5·10⁴`) thì chậm.

### 7.2 Cách 2b — Bỏ ký tự, khoá là `(vị trí, phần còn lại)`

Biến thể cùng ý tưởng: thay vì chèn `'*'`, dùng khoá `(i, word[:i] + word[i+1:])`. **Phải** có `i` trong khoá (xem lỗi 2 ở mục 13).

In [14]:
def removal_key(words):
    s = set()
    for word in words:
        for i in range(len(word)):
            key = (i, word[:i] + word[i + 1:])
            if key in s:
                return True
            s.add(key)
    return False

print([removal_key(w) for w, _ in examples])

[True, False, True]


### 7.3 Cách 3 — Rolling hash `O(n·m)` có xác minh va chạm

Từng đoạn:
1. Chuẩn bị `pw[k] = B^k mod P` cho `k < m`.
2. Với mỗi từ `w` (chỉ số `idx`), tính `H(w)` trong `O(m)`.
3. Với mỗi vị trí `j`: `h = (H − c_j · pw[m−1−j]) mod P`. Khoá `(j, h)`.
4. Tra `seen[(j, h)]` — danh sách chỉ số các từ trước đó có cùng khoá. Với mỗi ứng viên, **xác minh** bằng so sánh chuỗi `w[:j]+w[j+1:]` (hoặc so Hamming) để loại va chạm. Đúng thì trả `True`.
5. Thêm `idx` vào `seen[(j, h)]`.

Với `P = 2⁶¹ − 1` va chạm cực hiếm nên bước xác minh gần như không bao giờ chạy "oan"; kỳ vọng `O(n·m)`.

In [15]:
def rolling_hash(words, B=29, P=(1 << 61) - 1):
    if not words:
        return False
    m = len(words[0])
    pw = [1] * m
    for k in range(1, m):
        pw[k] = pw[k - 1] * B % P
    seen = defaultdict(list)
    for idx, w in enumerate(words):
        vals = [ord(ch) - 96 for ch in w]
        H = 0
        for v in vals:
            H = (H * B + v) % P
        for j in range(m):
            key = (j, (H - vals[j] * pw[m - 1 - j]) % P)
            for other in seen[key]:
                o = words[other]
                if o[:j] == w[:j] and o[j + 1:] == w[j + 1:]:    # xác minh, loại va chạm
                    return True
            seen[key].append(idx)
    return False

for words, want in examples:
    print(rolling_hash(words), "mong đợi", want)

True mong đợi True
False mong đợi False
True mong đợi True


Code nộp LeetCode cho cách 3. Để bước xác minh thực sự `O(1)` kỳ vọng, ta chỉ xác minh khi khoá trùng (rất hiếm khi trùng mà không đúng).

In [16]:
class SolutionHash:
    def differByOne(self, dict: List[str]) -> bool:
        words = dict
        if not words:
            return False
        B, P = 29, (1 << 61) - 1
        m = len(words[0])
        pw = [1] * m
        for k in range(1, m):
            pw[k] = pw[k - 1] * B % P
        seen = {}
        for idx, w in enumerate(words):
            vals = [ord(ch) - 96 for ch in w]
            H = 0
            for v in vals:
                H = (H * B + v) % P
            for j in range(m):
                key = (j, (H - vals[j] * pw[m - 1 - j]) % P)
                if key in seen:
                    for other in seen[key]:
                        o = words[other]
                        if sum(a != b for a, b in zip(o, w)) == 1:
                            return True
                    seen[key].append(idx)
                else:
                    seen[key] = [idx]
        return False

print([SolutionHash().differByOne(w) for w, _ in examples])

[True, False, True]


**Độ phức tạp:** Time kỳ vọng `O(n·m)` (mỗi khoá tính `O(1)`; xác minh `O(m)` chỉ xảy ra khi khoá trùng — hoặc là đáp án thật thì dừng ngay, hoặc là va chạm với xác suất ~`n·m / 2⁶¹`). Space `O(n·m)` số nguyên.

### 7.4 So sánh tốc độ cách 2 và cách 3 khi `m` lớn

Với 2 chuỗi dài `5·10⁴` (tổng `10⁵` ký tự), cách 2 phải tạo `10⁵` chuỗi dài `5·10⁴` — quá tốn. Ở đây đo với `m = 4000` cho vừa thời gian.

In [17]:
rng = random.Random(1554)
m = 4000
a = "".join(rng.choice(string.ascii_lowercase) for _ in range(m))
b = a[:-2] + ("z" if a[-2] != "z" else "y") + ("z" if a[-1] != "z" else "y")   # khác 2 ký tự cuối
for name, f in [("wildcard", wildcard_set), ("rolling", rolling_hash)]:
    t0 = time.perf_counter()
    r = f([a, b])
    print(f"{name:9s}: {r} ({time.perf_counter() - t0:.3f}s)")

wildcard : False (0.041s)
rolling  : False (0.011s)


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 3: `["abcd","cccc","abyd","abab"]`.

| Từ | i | Mẫu | Có trong set? |
|----|---|-----|---------------|
| abcd | 0..3 | `*bcd`, `a*cd`, `ab*d`, `abc*` | không (thêm cả 4) |
| cccc | 0..3 | `*ccc`, `c*cc`, `cc*c`, `ccc*` | không |
| abyd | 0 | `*byd` | không |
| abyd | 1 | `a*yd` | không |
| abyd | 2 | `ab*d` | **có** (từ `abcd`) ⇒ `True` |

In [18]:
def trace_wildcard(words):
    s = set()
    owner = {}
    for word in words:
        for i in range(len(word)):
            t = word[:i] + "*" + word[i + 1:]
            if t in s:
                print(f"  {word!r} i={i}: mẫu {t!r} ĐÃ CÓ (sinh bởi {owner[t]!r}) -> True")
                return True
            s.add(t)
            owner[t] = word
            print(f"  {word!r} i={i}: mẫu {t!r} mới, |set| = {len(s)}")
    print("  hết từ -> False")
    return False

trace_wildcard(["abcd", "cccc", "abyd", "abab"])

  'abcd' i=0: mẫu '*bcd' mới, |set| = 1
  'abcd' i=1: mẫu 'a*cd' mới, |set| = 2
  'abcd' i=2: mẫu 'ab*d' mới, |set| = 3
  'abcd' i=3: mẫu 'abc*' mới, |set| = 4
  'cccc' i=0: mẫu '*ccc' mới, |set| = 5
  'cccc' i=1: mẫu 'c*cc' mới, |set| = 6
  'cccc' i=2: mẫu 'cc*c' mới, |set| = 7
  'cccc' i=3: mẫu 'ccc*' mới, |set| = 8
  'abyd' i=0: mẫu '*byd' mới, |set| = 9
  'abyd' i=1: mẫu 'a*yd' mới, |set| = 10
  'abyd' i=2: mẫu 'ab*d' ĐÃ CÓ (sinh bởi 'abcd') -> True


True

In [19]:
trace_wildcard(["ab", "cd", "yz"])

  'ab' i=0: mẫu '*b' mới, |set| = 1
  'ab' i=1: mẫu 'a*' mới, |set| = 2
  'cd' i=0: mẫu '*d' mới, |set| = 3
  'cd' i=1: mẫu 'c*' mới, |set| = 4
  'yz' i=0: mẫu '*z' mới, |set| = 5
  'yz' i=1: mẫu 'y*' mới, |set| = 6
  hết từ -> False


False

Trace cách 3 (rolling hash) — in `H(w)` và khoá cho từng vị trí:

In [20]:
def trace_rolling(words, B=29, P=(1 << 61) - 1):
    m = len(words[0])
    pw = [pow(B, k, P) for k in range(m)]
    seen = {}
    for idx, w in enumerate(words):
        vals = [ord(c) - 96 for c in w]
        H = 0
        for v in vals:
            H = (H * B + v) % P
        print(f"từ {w!r}: H = {H}")
        for j in range(m):
            key = (j, (H - vals[j] * pw[m - 1 - j]) % P)
            hit = key in seen
            print(f"   j={j}: khoá = {key}{'  <-- trùng với ' + repr(words[seen[key]]) if hit else ''}")
            if hit:
                return True
            seen[key] = idx
    return False

trace_rolling(["abcd", "acbd", "aacd"])

từ 'abcd': H = 26162
   j=0: khoá = (0, 1773)
   j=1: khoá = (1, 24480)
   j=2: khoá = (2, 26075)
   j=3: khoá = (3, 26158)
từ 'acbd': H = 26974
   j=0: khoá = (0, 2585)
   j=1: khoá = (1, 24451)
   j=2: khoá = (2, 26916)
   j=3: khoá = (3, 26970)
từ 'aacd': H = 25321
   j=0: khoá = (0, 932)
   j=1: khoá = (1, 24480)  <-- trùng với 'abcd'


True

## 9. Code nộp LeetCode

Bản nộp chính: cách 2 (ngắn, đúng chữ ký doocs `differByOne(self, dict)`). Khi cần đảm bảo `O(n·m)` tuyệt đối, dùng `SolutionHash` ở mục 7.3.

In [21]:
class Solution:
    def differByOne(self, dict: List[str]) -> bool:
        s = set()
        for word in dict:
            for i in range(len(word)):
                t = word[:i] + "*" + word[i + 1 :]   # che vị trí i
                if t in s:                          # có từ trước khác đúng ở vị trí i
                    return True
                s.add(t)
        return False

for words, want in examples:
    got = Solution().differByOne(words)
    print(got, want)
    assert got == want

True True
False False
True True


## 10. Vì sao thuật toán đúng

**Bổ đề.** Với `u ≠ v` cùng độ dài: `u`, `v` khác nhau đúng 1 ký tự ⇔ tồn tại `i, j` sao cho `pattern(u, i) == pattern(v, j)`.

- (⇒) Nếu chỉ khác ở `i` thì `pattern(u, i) == pattern(v, i)`.
- (⇐) Hai mẫu bằng nhau thì vị trí ký tự `'*'` trùng nhau (chữ thường không chứa `'*'`) ⇒ `i = j`. Mọi vị trí khác `i` có ký tự giống nhau ⇒ `u`, `v` chỉ có thể khác ở `i`; vì `u ≠ v` nên khác **đúng** ở `i`.

**Thuật toán.** Mỗi mẫu được kiểm tra với mọi mẫu của các từ **trước đó** (đã nằm trong `set`). Mẫu của cùng một từ không trùng nhau. Vậy `True` được trả ⇔ có hai từ khác nhau sinh cùng mẫu ⇔ (bổ đề) có cặp khác đúng 1 ký tự. Nếu cặp `(u, v)` tồn tại với `v` đứng sau, khi xử lý `v` ở vị trí `i` thì mẫu của `u` đã có trong `set` ⇒ không bỏ sót.

**Rolling hash.** Khoá `(j, H − c_j·B^{m−1−j})` là hash đa thức của mẫu "che `j`"; mẫu bằng nhau ⇒ khoá bằng nhau (không bỏ sót). Khoá bằng nhau nhưng mẫu khác (va chạm) bị loại bởi bước xác minh ⇒ không báo nhầm.

In [22]:
# kiểm tra bổ đề bằng thực nghiệm trên mọi cặp chuỗi độ dài 3 trên bảng chữ 'ab'
from itertools import product
alls = ["".join(p) for p in product("ab", repeat=3)]
for u in alls:
    for v in alls:
        if u == v:
            continue
        share = any(u[:i] + "*" + u[i+1:] == v[:j] + "*" + v[j+1:] for i in range(3) for j in range(3))
        assert share == (hamming(u, v) == 1)
print("bổ đề đúng trên mọi cặp độ dài 3 bảng chữ {a,b}")

bổ đề đúng trên mọi cặp độ dài 3 bảng chữ {a,b}


## 11. Phân tích Time / Space

Gọi `n` = số chuỗi, `m` = độ dài mỗi chuỗi, với `n·m ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| So mọi cặp | `O(n²·m)` | `O(1)` | xấu nhất khi đáp án False |
| Wildcard + set | `O(n·m²)` | `O(n·m²)` | nhanh khi `m` nhỏ |
| Khoá (i, phần còn lại) | `O(n·m²)` | `O(n·m²)` | giống trên |
| Rolling hash | `O(n·m)` kỳ vọng | `O(n·m)` | tối ưu theo follow-up |

Thời gian băm chuỗi trong `set` là `O(độ dài)` — đó là lý do cách wildcard có thêm thừa số `m`. Rolling hash trả lời "average" `O(n·m)`; worst-case (đối thủ tạo va chạm có chủ đích) có thể tệ hơn nếu `B`, `P` cố định — có thể chọn `B` ngẫu nhiên khi chạy để phòng.

Thực tế trong Python: khi `m` nhỏ, cách wildcard thường **nhanh hơn** rolling hash vì slicing và băm chuỗi chạy bằng mã C, còn rolling hash là vòng lặp Python thuần. Rolling hash chỉ thắng rõ khi `m` lớn (thừa số `m²` bắt đầu đáng kể) — xem số đo dưới đây và ở mục 7.4.

In [23]:
rng = random.Random(11)
# đáp án False (trường hợp xấu nhất: phải xét hết), tổng số ký tự = 10^5
for n, m in [(20000, 5), (2000, 50), (100, 1000)]:
    words = no_pair_words(n, m, rng)
    for name, f in [("wildcard", wildcard_set), ("rolling", rolling_hash)]:
        t0 = time.perf_counter()
        r = f(words)
        print(f"n={n:>5}, m={m:>4}: {name:9s} -> {r} ({time.perf_counter() - t0:.3f}s)")

n=20000, m=   5: wildcard  -> False (0.031s)


n=20000, m=   5: rolling   -> False (0.094s)


n= 2000, m=  50: wildcard  -> False (0.033s)


n= 2000, m=  50: rolling   -> False (0.076s)


n=  100, m=1000: wildcard  -> False (0.144s)


n=  100, m=1000: rolling   -> False (0.109s)


## 12. Bộ test

In [24]:
METHODS = {
    "brute_pairs": brute_pairs,
    "wildcard_set": wildcard_set,
    "removal_key": removal_key,
    "rolling_hash": rolling_hash,
    "SolutionBrute": lambda w: SolutionBrute().differByOne(w),
    "SolutionWildcard": lambda w: SolutionWildcard().differByOne(w),
    "SolutionHash": lambda w: SolutionHash().differByOne(w),
    "Solution": lambda w: Solution().differByOne(w),
}
for words, want in examples:
    for name, f in METHODS.items():
        assert f(words) == want, (name, words)
print("ví dụ đề: OK")

ví dụ đề: OK


In [25]:
# edge case
edge = [
    (["a"], False),                 # chỉ 1 từ
    (["a", "b"], True),             # m = 1: mọi cặp khác đúng 1 ký tự
    (["ab", "ba"], False),          # khác 2 vị trí (bẫy cho cách bỏ ký tự không nhớ vị trí)
    (["abc", "abd"], True),         # khác ở cuối
    (["xbc", "abc"], True),         # khác ở đầu
    (["aaaa", "bbbb", "aabb"], False),
    (["abcde", "abfde", "zzzzz"], True),
]
for words, want in edge:
    for name, f in METHODS.items():
        assert f(words) == want, (name, words)
print("edge case: OK")

edge case: OK


In [26]:
# ngẫu nhiên có seed: đối chiếu với brute force
rng = random.Random(2020)
cnt_true = 0
for _ in range(2000):
    m = rng.randint(1, 6)
    alpha = rng.choice(["ab", "abc", "abcdef"])
    n = rng.randint(1, 15)
    words = list({"".join(rng.choice(alpha) for _ in range(m)) for _ in range(n)})
    ref = brute_pairs(words)
    cnt_true += ref
    for name, f in METHODS.items():
        assert f(words) == ref, (name, words)
print(f"2000 test ngẫu nhiên (seed 2020): OK, {cnt_true} test có đáp án True")

2000 test ngẫu nhiên (seed 2020): OK, 1360 test có đáp án True


In [27]:
# rolling hash với P nhỏ (cố tình gây va chạm) vẫn đúng nhờ bước xác minh
rng = random.Random(77)
for _ in range(1000):
    m = rng.randint(1, 6)
    words = list({"".join(rng.choice("abcdefgh") for _ in range(m)) for _ in range(rng.randint(1, 20))})
    assert rolling_hash(words, B=31, P=7) == brute_pairs(words)
print("rolling hash với P = 7 (va chạm dày đặc) vẫn đúng: OK")

rolling hash với P = 7 (va chạm dày đặc) vẫn đúng: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So một từ với chính nó

Vòng trong chạy `j` từ `0` thay vì `i + 1`, rồi kiểm tra "khác ≤ 1 ký tự" ⇒ từ nào cũng "khác 0 ký tự" với chính nó ⇒ luôn `True`.

In [28]:
def wrong_self_compare(words):
    for i in range(len(words)):
        for j in range(len(words)):                       # SAI: gồm cả j == i
            if sum(a != b for a, b in zip(words[i], words[j])) <= 1:   # SAI: <= 1
                return True
    return False

for words, want in examples:
    print(words, "| sai:", wrong_self_compare(words), "| đúng:", want)

['abcd', 'acbd', 'aacd'] | sai: True | đúng: True
['ab', 'cd', 'yz'] | sai: True | đúng: False
['abcd', 'cccc', 'abyd', 'abab'] | sai: True | đúng: True


Sửa: `for j in range(i + 1, n)` và điều kiện `== 1`.

### Lỗi 2 — Bỏ ký tự nhưng không ghi nhớ vị trí

Dùng khoá `word[:i] + word[i+1:]` trong một `set` chung cho mọi `i`: `"ab"` bỏ vị trí 0 → `"b"`, `"ba"` bỏ vị trí 1 → `"b"` ⇒ báo trùng, nhưng hai từ khác nhau ở **2** vị trí.

In [29]:
def wrong_no_position(words):
    s = set()
    for word in words:
        for i in range(len(word)):
            t = word[:i] + word[i + 1:]                   # SAI: mất thông tin vị trí
            if t in s:
                return True
            s.add(t)
    return False

for words in [["ab", "ba"], ["abc", "bca", "cab"], ["aab", "aba"]]:
    print(words, "| sai:", wrong_no_position(words), "| đúng:", brute_pairs(words))

['ab', 'ba'] | sai: True | đúng: False
['abc', 'bca', 'cab'] | sai: True | đúng: False
['aab', 'aba'] | sai: True | đúng: False


Sửa: dùng `'*'` giữ chỗ (`word[:i] + "*" + word[i+1:]`) hoặc khoá `(i, word[:i] + word[i+1:])`.

### Lỗi 3 — Thêm vào `set` trước rồi mới kiểm tra

`s.add(t)` đặt trước `if t in s` ⇒ điều kiện luôn đúng ngay mẫu đầu tiên.

In [30]:
def wrong_add_first(words):
    s = set()
    for word in words:
        for i in range(len(word)):
            t = word[:i] + "*" + word[i + 1:]
            s.add(t)                                      # SAI: thêm trước
            if t in s:
                return True
    return False

for words, want in examples:
    print(words, "| sai:", wrong_add_first(words), "| đúng:", want)

['abcd', 'acbd', 'aacd'] | sai: True | đúng: True
['ab', 'cd', 'yz'] | sai: True | đúng: False
['abcd', 'cccc', 'abyd', 'abab'] | sai: True | đúng: True


Sửa: kiểm tra `t in s` **trước**, rồi mới `s.add(t)`.

### Lỗi 4 — Rolling hash tin tuyệt đối vào hash (không xác minh) với modulo nhỏ

Hash chỉ là "dấu vân tay"; hai mẫu khác nhau có thể cùng hash. Với `P` nhỏ, va chạm xảy ra thường xuyên.

In [31]:
def wrong_hash_no_verify(words, B=31, P=7):
    m = len(words[0])
    seen = set()
    for w in words:
        vals = [ord(c) - 96 for c in w]
        H = 0
        for v in vals:
            H = (H * B + v) % P
        for j in range(m):
            key = (j, (H - vals[j] * pow(B, m - 1 - j, P)) % P)
            if key in seen:
                return True                               # SAI: không xác minh
            seen.add(key)
    return False

for words in [["ab", "cd", "yz"], ["abc", "xyz", "mnp", "qrs"]]:
    print(words, "| sai:", wrong_hash_no_verify(words), "| đúng:", brute_pairs(words))

['ab', 'cd', 'yz'] | sai: False | đúng: False
['abc', 'xyz', 'mnp', 'qrs'] | sai: True | đúng: False


Sửa: dùng `P` lớn (`2⁶¹ − 1`) **và** xác minh bằng so sánh chuỗi khi khoá trùng.

### Lỗi 5 — Đặt tên biến `dict` rồi gọi hàm `dict()` bên trong

Chữ ký doocs dùng tham số tên `dict`, che mất hàm có sẵn `dict`. Nếu bên trong lỡ viết `dict()` để tạo từ điển sẽ lỗi (gọi list như hàm). Ta bắt lỗi để minh hoạ, không để exception văng ra.

In [32]:
def wrong_shadow(dict):
    try:
        seen = dict()                                     # SAI: dict lúc này là list tham số
        return "chạy được"
    except TypeError as e:
        return f"TypeError: {e}"

print("sai:", wrong_shadow(["ab", "cd"]))

def fixed_shadow(dict):
    seen = {}                                             # ĐÚNG: dùng literal {} hoặc đổi tên words = dict
    words = dict
    return f"OK, {len(words)} từ, seen = {seen}"

print("đúng:", fixed_shadow(["ab", "cd"]))

sai: TypeError: 'list' object is not callable
đúng: OK, 2 từ, seen = {}


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Khác đúng 1 ký tự"**, **"xoá/đổi 1 ký tự"** ⇒ nghĩ ngay tới **mẫu wildcard** (`h*t`) — giống bài Word Ladder (LC 127) dùng `h*t` để nối các từ kề nhau.
- Ràng buộc là **tổng số ký tự** ⇒ phân tích theo `n·m`; cẩn thận thừa số `m` ẩn trong việc băm chuỗi.
- Cần `O(n·m)` nghiêm ngặt ⇒ **hash đa thức**, "bỏ ký tự `j`" = trừ một số hạng.
- Luôn xác minh khi dùng hash (hoặc dùng `P` cực lớn + chấp nhận xác suất).
- Kiểm tra-rồi-thêm trong cùng vòng chỉ an toàn khi các khoá của **cùng một phần tử** không thể trùng nhau.

## 15. Biến thể của bài

1. **Đếm số cặp** khác đúng 1 ký tự (bài tập 5): đếm tần suất mỗi mẫu, cộng `C(k, 2)`.
2. **Trả về cặp cụ thể** (bài tập 4).
3. **Khác tối đa `k` ký tự** — khó hơn nhiều; với `k` nhỏ có thể dùng nguyên lý chuồng bồ câu (chia chuỗi thành `k+1` khối).
4. **Implement Magic Dictionary (LC 676)**: tra một từ có khác đúng 1 ký tự với từ nào trong từ điển — dùng đúng kỹ thuật wildcard.
5. **Word Ladder (LC 127)**: BFS trên đồ thị "khác 1 ký tự", dựng cạnh bằng mẫu wildcard.
6. Chuỗi **khác độ dài**, cho phép thêm/xoá 1 ký tự (One Edit Distance, LC 161).

In [33]:
# Biến thể 4 (Magic Dictionary) bằng wildcard: đếm số từ sinh ra mỗi mẫu + nhớ từ gốc
class MagicDictionary:
    def buildDict(self, words):
        self.cnt = Counter()
        self.words = set(words)
        for w in words:
            for i in range(len(w)):
                self.cnt[w[:i] + "*" + w[i + 1:]] += 1

    def search(self, w):
        for i in range(len(w)):
            k = self.cnt[w[:i] + "*" + w[i + 1:]]
            if k > 1 or (k == 1 and w not in self.words):
                return True
        return False

md_ = MagicDictionary()
md_.buildDict(["hello", "leetcode"])
print([md_.search(x) for x in ["hello", "hhllo", "hell", "leetcoded"]], "(LC 676 cho [False, True, False, False])")

[False, True, False, False] (LC 676 cho [False, True, False, False])


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. So mọi cặp | Hamming từng cặp | `O(n²·m)` | `O(1)` | kiểm chứng, `n` nhỏ |
| 2. Wildcard + set | che từng vị trí, trùng mẫu ⇒ True | `O(n·m²)` | `O(n·m²)` | **bản nộp** khi `m` nhỏ (doocs) |
| 2b. Khoá (i, phần còn lại) | như 2, không dùng `'*'` | `O(n·m²)` | `O(n·m²)` | khi bảng chữ có thể chứa `'*'` |
| 3. Rolling hash | hash mẫu trong `O(1)` | `O(n·m)` | `O(n·m)` | follow-up, `m` lớn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `differByOne(["abc","abd","xyz"])` và `differByOne(["abc","bca","cab"])` trả về gì?

**Bài 2 (tính tay):** Liệt kê các mẫu wildcard sinh ra khi chạy cách 2 trên `["ab","cb","ca"]` theo thứ tự, và cho biết thuật toán dừng ở mẫu nào.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_hamming(a, b)` trả số vị trí khác nhau của hai chuỗi cùng độ dài.

**Bài 4 (tự viết lại):** Viết `ex4_find_pair(words)` trả về cặp chỉ số `(i, j)` với `i < j` sao cho `words[i]`, `words[j]` khác đúng 1 ký tự, hoặc `None` nếu không có. Yêu cầu dùng `dict` mẫu → chỉ số, `O(n·m²)`.

**Bài 5 (mini challenge):** Viết `ex5_count_pairs(words)` trả về **số cặp** (không thứ tự) khác đúng 1 ký tự.

In [34]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_hamming(a, b):
    diff = 0
    for x, y in zip(a, b):
        # TODO: tăng diff nếu x khác y
        pass
    return diff

In [35]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_find_pair(words):
    # TODO
    raise NotImplementedError

def ex5_count_pairs(words):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tìm hai từ chung tiền tố dài. (2) `"abc"`/`"abd"` khác 1 ký tự; trong `["abc","bca","cab"]` mọi cặp khác 3 ký tự. (3) `True`, `False`.

**Bài 2.** (1) `"ab"` sinh `*b`, `a*`. (2) `"cb"` sinh `*b` — đã có! (3) Dừng ngay ở mẫu `*b` của `"cb"`, trả `True` (không xét `"ca"`).

**Bài 3.** (1) So từng cặp ký tự. (2) `if x != y: diff += 1`. (3) Hoặc `sum(x != y for x, y in zip(a, b))`.

**Bài 4.** (1) Thay `set` bằng `dict` `mẫu → chỉ số từ đầu tiên sinh mẫu`. (2) Gặp mẫu đã có ⇒ trả `(first[t], j)`. (3) Duyệt `j` tăng dần nên `first[t] < j`.

**Bài 5.** (1) Mỗi cặp khác đúng 1 ký tự chia sẻ **đúng một** mẫu. (2) Đếm `Counter` các mẫu. (3) Đáp án `Σ k·(k−1)/2` trên mọi mẫu.

## 19. Đáp án tham khảo cho bài tập

In [36]:
# Bài 1
a1 = (Solution().differByOne(["abc", "abd", "xyz"]), Solution().differByOne(["abc", "bca", "cab"]))
print("Bài 1:", a1)
assert a1 == (True, False)

# Bài 2
print("Bài 2:")
assert trace_wildcard(["ab", "cb", "ca"]) is True

Bài 1: (True, False)
Bài 2:
  'ab' i=0: mẫu '*b' mới, |set| = 1
  'ab' i=1: mẫu 'a*' mới, |set| = 2
  'cb' i=0: mẫu '*b' ĐÃ CÓ (sinh bởi 'ab') -> True


In [37]:
# Bài 3
def ex3_hamming_sol(a, b):
    diff = 0
    for x, y in zip(a, b):
        if x != y:
            diff += 1
    return diff

assert ex3_hamming_sol("abcd", "aacd") == 1
assert ex3_hamming_sol("abc", "xyz") == 3
assert ex3_hamming_sol("same", "same") == 0
print("Bài 3: OK")

Bài 3: OK


In [38]:
# Bài 4
def ex4_find_pair_sol(words):
    first = {}
    for j, w in enumerate(words):
        for i in range(len(w)):
            t = w[:i] + "*" + w[i + 1:]
            if t in first:
                return (first[t], j)
            first[t] = j
    return None

rng4 = random.Random(4)
for _ in range(800):
    m = rng4.randint(1, 5)
    words = list({"".join(rng4.choice("abc") for _ in range(m)) for _ in range(rng4.randint(1, 10))})
    res = ex4_find_pair_sol(words)
    if res is None:
        assert not brute_pairs(words)
    else:
        i, j = res
        assert i < j and ex3_hamming_sol(words[i], words[j]) == 1
print("Bài 4: OK, ví dụ:", ex4_find_pair_sol(["abcd", "cccc", "abyd", "abab"]))

Bài 4: OK, ví dụ: (0, 2)


In [39]:
# Bài 5
def ex5_count_pairs_sol(words):
    cnt = Counter()
    for w in words:
        for i in range(len(w)):
            cnt[w[:i] + "*" + w[i + 1:]] += 1
    return sum(k * (k - 1) // 2 for k in cnt.values())

def ex5_brute(words):
    n = len(words)
    return sum(ex3_hamming_sol(words[i], words[j]) == 1 for i in range(n) for j in range(i + 1, n))

rng5 = random.Random(5)
for _ in range(800):
    m = rng5.randint(1, 5)
    words = list({"".join(rng5.choice("abc") for _ in range(m)) for _ in range(rng5.randint(1, 12))})
    assert ex5_count_pairs_sol(words) == ex5_brute(words), words
print("Bài 5: OK, ví dụ:", ex5_count_pairs_sol(["abcd", "acbd", "aacd"]), ex5_count_pairs_sol(["a", "b", "c"]))

Bài 5: OK, ví dụ: 1 3


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def differByOne(self, dict: List[str]) -> bool:
        s = set()
        for word in dict:
            for i in range(len(word)):
                t = word[:i] + "*" + word[i + 1:]
                if t in s:
                    return True
                s.add(t)
        return False
```

**Template "khác đúng 1 ký tự":** sinh mẫu wildcard cho mỗi (từ, vị trí) → tra/gom trong `set`/`Counter`/`dict`. Cần `O(n·m)`: hash đa thức, khoá `(j, H − c_j·B^{m−1−j})`, xác minh khi trùng.

**Checklist trước khi submit:**
- [ ] Mẫu **giữ chỗ** vị trí bị che (`'*'` hoặc khoá có `i`).
- [ ] Kiểm tra `in` **trước** `add`.
- [ ] Không so một từ với chính nó; điều kiện là **đúng 1** khác biệt.
- [ ] Không gọi `dict()` khi tham số tên `dict` (dùng `{}` hoặc đổi tên).
- [ ] Rolling hash: `P` lớn + xác minh va chạm.
- [ ] Thử nhanh: ví dụ 1–3, `["a"] → False`, `["ab","ba"] → False`, `["a","b"] → True`.